## GenAI Disclosure Statement

Generative AI tools were used as learning aids. All analysis and conclusions are the team's own work.

---

# Notebook 07 — Final Audit Summary and Deployment Recommendation
### DNSC 6330 Responsible Machine Learning Capstone | GWU

**Purpose:** Integrate all evidence from Notebooks 01–06 into a coherent audit narrative.
Score each evidence dimension. Produce the final deployment recommendation.

**Lecture 06 connection:** Defensibility = documented objective + measured disparities + tested
robustness + known residual risks + monitoring plan. This notebook is the capstone integration point.


In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), ".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")
import warnings
warnings.filterwarnings("ignore")

TABLES_DIR = os.path.join(os.getcwd(), "..", "tables")
DOCS_DIR   = os.path.join(os.getcwd(), "..", "docs")
FIG_DIR    = os.path.join(os.getcwd(), "..", "figures")


## Section 7.1 — Evidence Crosswalk

Maps each of the five capstone questions to the artifacts that answer them.
This crosswalk is the audit backbone.


In [ ]:
crosswalk = pd.DataFrame([
    {
        "Capstone Question": "Q1: What is the system, who is affected, what is failure?",
        "Lecture": "Lec 01",
        "Evidence Artifact": "docs/00_system_card.md, harm matrix, decision log D-001",
        "Notebook": "Phase 0 / System Card",
        "Status": "Complete",
    },
    {
        "Capstone Question": "Q2: What is the model learning? Any proxy risks?",
        "Lecture": "Lec 02",
        "Evidence Artifact": "tables/top_features_shap.csv, proxy_correlation.csv, counterfactuals.csv, figures/shap_*.png",
        "Notebook": "Notebook 03",
        "Status": "Complete",
    },
    {
        "Capstone Question": "Q3: Are outcomes equitable across protected groups?",
        "Lecture": "Lec 03",
        "Evidence Artifact": "tables/master_fairness_table.csv, intersectional_table.csv, figures/air_by_threshold.png, intersectional_heatmap.png",
        "Notebook": "Notebook 04",
        "Status": "Complete",
    },
    {
        "Capstone Question": "Q4: Is the model robust and stable? How do we detect failure?",
        "Lecture": "Lec 04",
        "Evidence Artifact": "tables/psi_table.csv, review_trigger_table.csv, figures/calibration_*.png",
        "Notebook": "Notebook 05",
        "Status": "Complete",
    },
    {
        "Capstone Question": "Q5: What is the threat surface? What residual risks remain?",
        "Lecture": "Lec 05",
        "Evidence Artifact": "tables/threat_model_table.csv, access_control_matrix.csv, docs/06_security_residual_risk.md",
        "Notebook": "Notebook 06",
        "Status": "Complete",
    },
])
display(crosswalk)
crosswalk.to_csv(os.path.join(TABLES_DIR, "evidence_crosswalk.csv"), index=False)


## Section 7.2 — Model Performance Summary

In [ ]:
metrics_df = pd.read_csv(os.path.join(TABLES_DIR, "metrics_table_final.csv"))
print("Final Model Performance (Test Set):")
display(metrics_df[["model", "split", "threshold", "n", "auc", "pr_auc", "ks", "brier", "f1"]])


## Section 7.3 — Fairness Summary

In [ ]:
ft = pd.read_csv(os.path.join(TABLES_DIR, "master_fairness_table.csv"))
non_ref = ft[~ft["is_reference"]]
below_080 = non_ref[non_ref["air"] < 0.80]

print("AIR Summary — All Protected Groups:")
display(ft[["attribute", "group", "is_reference", "n", "approval_rate", "air", "fnr", "ece"]])

if len(below_080) > 0:
    print("\n⚠ GROUPS WITH AIR < 0.80:")
    display(below_080[["attribute", "group", "n", "approval_rate", "air", "fnr"]])
else:
    print("\n All measured groups have AIR >= 0.80 at the operating threshold.")


## Section 7.4 — Robustness Summary

In [ ]:
psi_df = pd.read_csv(os.path.join(TABLES_DIR, "psi_table.csv"))
print("PSI Summary — Training vs. Geographic Holdout (CA):")
display(psi_df[["feature", "psi", "interpretation", "action"]])

major_shift = psi_df[psi_df["psi"] > 0.25]
if len(major_shift) > 0:
    print("\n⚠ FEATURES WITH MAJOR SHIFT (PSI > 0.25):")
    display(major_shift[["feature", "psi", "interpretation"]])
else:
    print("\n No features with major distribution shift (PSI > 0.25).")


## Section 7.5 — Residual Risk Register

In [ ]:
residual_risk = pd.DataFrame([
    {"ID": "RR-001", "Risk": "Geographic/census features may proxy for race",
     "Severity": "High", "Likelihood": "Medium", "Status": "Open — resolved after SHAP analysis"},
    {"ID": "RR-002", "Risk": "Small intersectional cells (AIAN, NHPI)",
     "Severity": "Medium", "Likelihood": "Certain", "Status": "Accepted — documented caveat"},
    {"ID": "RR-003", "Risk": "Single training year (2024 LAR only)",
     "Severity": "Medium", "Likelihood": "Certain", "Status": "Accepted — geographic holdout tested"},
    {"ID": "RR-004", "Risk": "Broker gaming: income/DTI inflation",
     "Severity": "Medium", "Likelihood": "High", "Status": "Open — plausibility checks recommended"},
    {"ID": "RR-005", "Risk": "Calibration gap between racial subgroups",
     "Severity": "Medium", "Likelihood": "Medium", "Status": "Open — quarterly monitoring"},
    {"ID": "RR-006", "Risk": "Third-party vendor data poisoning",
     "Severity": "Medium", "Likelihood": "Low", "Status": "Open — hash validation required"},
    {"ID": "RR-007", "Risk": "Protected attributes may not be captured at scoring time",
     "Severity": "High", "Likelihood": "Medium", "Status": "Conditional — deployment condition"},
])
display(residual_risk)
residual_risk.to_csv(os.path.join(TABLES_DIR, "residual_risk_summary.csv"), index=False)


## Section 7.6 — Deployment Decision Scorecard

In [ ]:
# Build scorecard from loaded evidence
try:
    metrics_row = metrics_df[metrics_df["model"].str.contains("GBM")].iloc[0]
    auc_val = float(metrics_row["auc"])
    perf_score = "Green" if auc_val >= 0.72 else ("Yellow" if auc_val >= 0.68 else "Red")
except:
    auc_val = None; perf_score = "Unknown"

try:
    min_air = non_ref["air"].min()
    fair_score = "Green" if min_air >= 0.80 else ("Yellow" if min_air >= 0.70 else "Red")
except:
    min_air = None; fair_score = "Unknown"

try:
    max_psi = psi_df["psi"].max()
    robust_score = "Green" if max_psi < 0.10 else ("Yellow" if max_psi < 0.25 else "Red")
except:
    max_psi = None; robust_score = "Unknown"

scorecard = pd.DataFrame([
    {"Dimension": "Performance (AUC)", "Key Finding": f"AUC = {auc_val}", "Score": perf_score},
    {"Dimension": "Fairness — AIR", "Key Finding": f"Min AIR = {min_air}", "Score": fair_score},
    {"Dimension": "Fairness — Error-rate parity", "Key Finding": "See master fairness table", "Score": "See NB04"},
    {"Dimension": "Calibration (overall + subgroup)", "Key Finding": "See calibration tables", "Score": "See NB05"},
    {"Dimension": "Explainability / Proxy risk", "Key Finding": "See top_features_shap.csv", "Score": "See NB03"},
    {"Dimension": "Robustness — PSI + drift", "Key Finding": f"Max PSI = {max_psi}", "Score": robust_score},
    {"Dimension": "Security — threat coverage", "Key Finding": "5 scenarios; 2 unmitigated", "Score": "Yellow"},
    {"Dimension": "Documentation completeness", "Key Finding": "System card, model card, decision log, risk register", "Score": "Green"},
])
display(scorecard)
scorecard.to_csv(os.path.join(TABLES_DIR, "deployment_scorecard.csv"), index=False)


## Section 7.7 — Deployment Recommendation

Based on the evidence scorecard, produce the final deployment recommendation.
See `docs/07_deployment_recommendation.md` for the full recommendation memo.


In [ ]:
print("=" * 65)
print("DEPLOYMENT RECOMMENDATION — FINAL AUDIT JUDGMENT")
print("=" * 65)
print()

# Count scores
score_counts = scorecard["Score"].value_counts()
n_red    = score_counts.get("Red", 0)
n_yellow = score_counts.get("Yellow", 0)
n_green  = score_counts.get("Green", 0)

print(f"Scorecard: {n_green} Green | {n_yellow} Yellow | {n_red} Red")
print()

if n_red == 0 and n_yellow <= 2:
    recommendation = "DEPLOY WITH CONDITIONS"
    rationale = (
        "Evidence is sufficiently strong to support deployment within defined safeguards. "
        "All conditions in docs/07_deployment_recommendation.md must be satisfied before production use."
    )
elif n_red == 0 and n_yellow <= 4:
    recommendation = "DO NOT DEPLOY YET"
    rationale = (
        "Multiple dimensions require improvement before deployment. "
        "Address Yellow items and re-evaluate."
    )
else:
    recommendation = "PAUSE AND REDESIGN"
    rationale = "Critical findings require model or process redesign before re-evaluation."

print(f"  RECOMMENDATION: {recommendation}")
print()
print(f"  Rationale: {rationale}")
print()
print("Pause / Reverse Conditions:")
print("  1. AIR < 0.80 for any protected group for 2 consecutive monitoring periods  IMMEDIATE PAUSE")
print("  2. PSI > 0.25 for loan_amount or income  Pause retraining; feature review")
print("  3. FNR gap widens > 20% vs. baseline for any minority group  Escalate to compliance")
print("  4. New top-5 SHAP feature with |r| > 0.30 vs. any protected attribute  Fairness re-audit")
print("  5. Training data integrity compromise  IMMEDIATE PAUSE; full audit")
print()
print("Full recommendation: docs/07_deployment_recommendation.md")
print()
print("=" * 65)
print("AUDIT RECORD COMPLETE")
print("Three-pillar check:")
print("  [] Measurement before opinion     fairness table built before recommendation")
print("  [] Diagnostics before remediation  proxy + robustness analysis before scorecard")
print("  [] Documentation before deployment  system card, decision log, risk register complete")
print("=" * 65)
